In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")

llm = ChatOpenAI(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)

In [3]:
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

tools = [add_numbers]

In [4]:
# 创建 React 代理
agent = create_agent(model=llm, tools=tools)
# 执行任务
result = agent.invoke({"messages": [("human", "请计算 3.5 + 2.7")]})
print("答案：", result["messages"][-1].content)

答案： 3.5 + 2.7 = **6.2**


In [5]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks
into a style that is {style}.
text: ```{customer_email}```
"""
print(prompt)

Translate the text that is delimited by triple backticks
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [6]:
result = agent.invoke({"messages": [("human", "请计算 3.5 + 2.7")]})
print("答案：", result["messages"])
print("答案：", result["messages"][-1].content)

答案： [HumanMessage(content='请计算 3.5 + 2.7', additional_kwargs={}, response_metadata={}, id='108e7d75-22e0-4076-8eab-e54acb860d34'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 292, 'total_tokens': 355, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 36}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '9b466dd7-af93-4048-b600-d6c507e1c604', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3ac8-0bef-7e12-bfc3-c93754619274-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 3.5, 'b': 2.7}, 'id': 'call_00_IYApBrBj5BFAWdRa7Qa24928', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 292, 'output_tokens': 63, 'total_tokens': 355, 'input_token_details':

In [7]:
template_string = """Translate the text \
that is delimited by triple backticks into a style that is {style}. \
text: ```{customer_email}```
"""

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [12]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['customer_email', 'style'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{customer_email}```\n')

In [13]:
customer_style = """American English \
in a calm and respectful tone"""

In [17]:
customer_email = """Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [25]:
customer_messages = prompt_template.format_messages(style=customer_style, customer_email=customer_email)

In [26]:
print(type(customer_message))
print(type(customer_message[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [36]:
print(customer_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone. text: ```Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [29]:
print(customer_messages)

[HumanMessage(content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone. text: ```Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n", additional_kwargs={}, response_metadata={})]


In [31]:
customer_response = agent.invoke({"messages": [("human", customer_messages[0].content)]})

In [37]:
print(customer_response)

{'messages': [HumanMessage(content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone. text: ```Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n", additional_kwargs={}, response_metadata={}, id='e536de7b-9131-407c-b972-4cd714799f01'), AIMessage(content="Of course. Here is your text translated into a calm and respectful American English style:\n\nI'm quite frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up the mess. I would appreciate your help with this matter.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 363, 'total_tokens': 428, 'completion_tokens_details': 

In [38]:
print(customer_response['messages'][-1].content)

Of course. Here is your text translated into a calm and respectful American English style:

I'm quite frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up the mess. I would appreciate your help with this matter.


In [39]:
service_reply = """Hey there matey! I understand that you're upset about the blender lid flying off and causing a mess in your kitchen. I apologize for the inconvenience this has caused you. While the warranty may not cover the cost of cleaning up, I'm here to help you find a solution. Could you please provide me with more details about the incident, such as when it happened and if there were any injuries? This information will help me assist you better. Thank you for your patience!"""

In [40]:
service_style_pirate = """a polite tone that speaks in English Pirate"""

In [44]:
customer_messages = prompt_template.format_messages(style=service_style_pirate, customer_email=service_reply)
customer_response = agent.invoke({"messages": [("human", customer_messages[0].content)]})
print(customer_response['messages'][-1].content)

Ahoy there, me hearty! I be understandin' that ye be feelin' a bit o' the doldrums 'bout the blender lid takin' flight and makin' a right mess o' yer galley. I do apologize fer the trouble this has brought upon ye. Though the warranty may not be coverin' the cost o' swabbin' the decks, I be here to help ye find a course o' action. Could ye kindly provide me with more details o' the incident, such as when it happened and if any hands were injured? This here information will help me serve ye better. Thank ye fer yer patience, and may fair winds fill yer sails!
